# **Travaux exploratoires : résumé formaté d'un texte**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 22/07/2025*

**But :** à partir d'un texte, fournir en sortie un dictionnaire du type {evenement:entites} avec :

- **evenement :** une chaine de caravctères décrivant de façon très synthétique un événement relaté dans le texte
- **lieu :** où cet événement a eu lieu
- **temps :** quand cet événement a eu lieu
- **individus :** quels sont les individus impliqués dans l'événement

La longueur de ce dictionnaire doit être égale au nombre d'événements relatés dans le texte. Chaque événement apparait une et une seule fois dans le dictionnaire. 

Il s'agit donc, de relever chaque événement d'un texte et, pour chaque événement, d'apporter une réponse aux questions **quoi ? où ? quand ? qui ?**

**Note :** ces travaux exploratoires sont le fruit d'une collaboration avec ChatGPT et Copilot.

## **Le texte utilisé pour les expérimentations**

**Note :** ce texte a été généré par ChatGPT 4o

**Prompt :** *peux tu générer un compte-rendu d'enquête de police d'une vingtaine de lignes stp ?*

In [40]:
texte = """
Compte-rendu d’enquête – Rapport préliminaire

Objet : Enquête sur des activités suspectes signalées dans le quartier des Érables (Secteur 5)

Date : 24 juillet 2025
Responsable de l’enquête : Lieutenant A. Mercier
Durée de l’enquête : Du 18 au 24 juillet 2025

Suite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte. Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives. Les agents ont constaté la présence récurrente de véhicules immatriculés hors département, ainsi que des échanges brefs entre occupants et des individus arrivant à pied.

Une perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet. Sur place, les enquêteurs ont découvert plusieurs sachets contenant une substance poudreuse suspectée d’être de la cocaïne (analyse en cours), ainsi qu’un total de 3 200 € en espèces, deux téléphones portables et un carnet mentionnant des noms et montants.

Trois personnes ont été interpellées sur place et placées en garde à vue. L’une d’entre elles est connue des services pour trafic de stupéfiants. Les auditions sont toujours en cours.

L’enquête se poursuit pour établir les filières d’approvisionnement et identifier d’éventuels complices.
"""

## **Le but visé : la réponse idéale de ChatGPT... sans ChatGPT**

**Voici un prompt envoyé à ChatGPT 4o.**

**Prompt :** 

*Si je te demande de repérer dans ce texte tous les événements significatifs dans le cadre d'une enquête de police, et de me fournir une réponse formatée du type :*

*evenement : un evenement significatif dans le cadre d'une enquête de police*

*lieu : le lieu de l'évenement*

*moment : le moment où a eu lieu cet événement*

*qui : les individus impliqués dans cet événement*

*que me réponds-tu ?*

**Sa réponse :**


**événement** : Signalement d'activités suspectes par les riverains

**lieu** : 17 rue des Marronniers, quartier des Érables (Secteur 5)

**moment** : avant le 18 juillet 2025

**qui** : riverains du quartier

_______________________________________________________________________________

**événement**  : Mise en place d’une surveillance discrète

**lieu :** 17 rue des Marronnier

**moment** : du 18 au 23 juillet 2025 (cinq nuits)

**qui** : agents enquêteurs


_______________________________________________________________________________


**événement**  : Observation de véhicules suspects et d’échanges brefs

**lieu** : 17 rue des Marronniers

**moment** : pendant la période de surveillance (18–23 juillet 2025)

**qui** : occupants des véhicules hors département et individus à pied


_______________________________________________________________________________


**événement**  : Perquisition du domicile

**lieu** : 17 rue des Marronniers

**moment** : 23 juillet 2025 à 6h00

**qui** : officiers de police judiciaire, enquêteurs


_______________________________________________________________________________


**événement**  : Découverte de stupéfiants présumés et d'objets suspects

**lieu** : 17 rue des Marronniers

**moment** : lors de la perquisition, 23 juillet 2025

**qui** : enquêteurs


_______________________________________________________________________________


**événement**  : Interpellation de trois personnes

**lieu** : 17 rue des Marronniers

**moment** : 23 juillet 2025

**qui** : trois suspects (dont un connu des services)


_______________________________________________________________________________


**événement**  : Placement en garde à vue

**lieu** : non précisé (présumément commissariat local)

**moment** : à partir du 23 juillet 2025

**qui** : les trois individus interpellés


_______________________________________________________________________________



**événement**  : Auditions en cours

**lieu** : non précisé

**moment** : au 24 juillet 2025

**qui** : suspects interpellés, enquêteurs

## **Approche 1 : Flan-T5 (par Google) avec spécialisation par few-shot + spaCy**

**Flan-T5 =** ensemble de modèles de langage open source développés par Google, basés sur l’architecture T5 (Text‑to‑Text Transfer Transformer), enrichis d'une opération  de fine-tuning.

**Avantages de Flan-T5 :**

- très bon pour instructions, multilingue
- taille modérée : 770M ou 3B (selon RAM GPU/CPU)
- open source, disponible via HuggingFace
- bien adapté pour des tâches d’instruction (extraction, résumé, question-réponse)
- pas franco-français mais français bien supporté

**Inconvénient :** 
- demande un GPU moyen ou un CPU assez puissant (exemple 770M peut tourner CPU lentement).

**<u>1.a. Paramètres du modèle Flan-T5</u>**

- **max_length =** le nombre de tokens au-delà duquel l'entrée est tronquée
- **max_new_tokens =** le nombre de tokens au-delà duquel la sortie est tronquée

Ce modèle a une limite réelle de 512 tokens au total, autrement dit :

$$max\_length+max\_new\_tokens\leq 512$$

Le non-respect de ces limites dans le paramétrage du modèle entraîne des textes coupés au niveau de l'entrée et/ou de la sortie, voire des textes vides.

**Pour optimiser les traitements, il est important de contrôler ces paramètres, mais aussi le prompt lui-même !**

**NOTE 1 (26/07/2025) :** à cette date, ChatGPT (gratuit) fait ça encore assez mal, c'est donc une tâche qui pour le moment est encore complètement dévolue à l'humain. Mais les choses peuvent changer rapidement !

**NOTE 2 : 1 mot $\approx$ 1,5 à 1,6 tokens** autrement dit **1 token $\approx$ 0,65 mot**.

Ainsi : **512 tokens $\approx$ 333 mots**

**<u>1.b. Un premier exemple de prompt</u>**

Dans ce qui suit, la spécialisation du modèle se fait par **few-shot**, autrement dit on ajoute des exemples dans le prompt. Cette approche présente l'avantage d'être plus simple à mettre en oeuvre que le fine-tuning.

In [307]:
prompt = f"""
Tu vas lire ce compte-rendu d'enquête. Ton objectif est d’en extraire les événements décrits.
Pour chaque événement significatif dans le cadre d'une enquête de police, donne-moi une réponse structurée ainsi :

- événement : [résumé en une phrase courte]
- où : [lieux concernés]
- quand : [dates ou périodes]
- qui : [personnes ou groupes impliqués]

Exemple :

Texte :
Le 3 juin 2023, un incendie a été signalé dans un entrepôt à Lyon. Les pompiers sont intervenus vers 23h.

Réponse :
1. 
- événement : Incendie dans un entrepôt
- où : Lyon
- quand : 3 juin 2023, vers 23h
- qui : pompiers

Voici le texte :
{texte}

Retourne ta réponse en texte clair, sous forme de liste numérotée.
"""

In [309]:
prompt

"\nTu vas lire ce compte-rendu d'enquête. Ton objectif est d’en extraire les événements décrits.\nPour chaque événement significatif dans le cadre d'une enquête de police, donne-moi une réponse structurée ainsi :\n\n- événement : [résumé en une phrase courte]\n- où : [lieux concernés]\n- quand : [dates ou périodes]\n- qui : [personnes ou groupes impliqués]\n\nExemple :\n\nTexte :\nLe 3 juin 2023, un incendie a été signalé dans un entrepôt à Lyon. Les pompiers sont intervenus vers 23h.\n\nRéponse :\n1. \n- événement : Incendie dans un entrepôt\n- où : Lyon\n- quand : 3 juin 2023, vers 23h\n- qui : pompiers\n\nVoici le texte :\n\nCompte-rendu d’enquête – Rapport préliminaire\n\nObjet : Enquête sur des activités suspectes signalées dans le quartier des Érables (Secteur 5)\n\nDate : 24 juillet 2025\nResponsable de l’enquête : Lieutenant A. Mercier\nDurée de l’enquête : Du 18 au 24 juillet 2025\n\nSuite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabitu

**A priori plutôt pas mal... tant qu'on ne regarde pas son nombre de tokens !**

In [148]:
encoded = tokenizer(prompt, return_tensors="pt", truncation=False)
prompt_len = encoded["input_ids"].shape[1]
print("Taille du prompt (tokens) :", prompt_len)

Taille du prompt (tokens) : 663


**Rappel :** on doit avoir 

$$nb\_tokens\_prompt+nb\_tokens\_reponse \leq 512$$

**<u>1.c. Solution naturelle : découper le texte en plusieurs morceaux</u>**

**Tout d'abord, on alloue un certain nombre de tokens à la sortie**

On part d'une sortie type pour 1 événement et on compte son nombre de tokens :

In [178]:
sortie_type_evenement = """
événement : Interpellation de trois personnes

lieu : 17 rue des Marronniers

moment : 23 juillet 2025

qui : trois suspects (dont un connu des services)
"""

encoded_sortie_type_evenement = tokenizer(sortie_type_evenement, return_tensors="pt", truncation=False)
sortie_type_evenement_len = encoded_sortie_type_evenement["input_ids"].shape[1]
print("Taille d'une sortie type (tokens) :", sortie_type_evenement_len)

Taille d'une sortie type (tokens) : 44


**Analyse de la situation :** 

- il y a 44 tokens dans la sortie type  
- on vise un peu large et on décide d'allouer **80 tokens en sortie**, ce qui laisse **512-80 = 432 tokens en entrée** 
- **Problème :** une entrée de 432 tokens peut potentiellement recouvrir **plusieurs** événements. Et dans ce cas, le nombre de tokens en sortie risque d'être insuffisant- 
- **Idée 1 :** limiter le nombre de tokens en entrée au nombre de tokens tout juste nécessaire pour détecter UN événement, mais comment faire ça ?
- **Idée 2 :** essayer de déterminer une relation sur le nombre de tokens en sortie et le nombre de tokens en entrée lorqu'on s'intéresse à **UN événement**... mais comment faire ça (stats ?)

Tout le problème est donc de répondre à la question suivante :

**<center>Comment découper intelligemment un texte en segments ?</center>**

On aimerait pour cela pouvoir répondre à la question :

*<center>1 évenement = combien de tokens en entrée ?</center>*

Mais la réponse est très fluctuante, elle dépend :

- de l'événement lui-même
- de la façon de le rédiger (et donc en particulier de l'enquêteur)
- etc.

**Fixer un seuil de tokens par segment sur l'entrée**

On prend par exemple 

$$s_{entree}=200$$

C'est empirique, on pourra tester d'autres valeurs, mais on se dit qu'avec 200 tokens il y a assez peu de chances qu'il y ait plus qu'un événement, et donc on espère générer ainsi des segments avec 0 ou 1 événement.

**<u>1.d.Mise en oeuvre d'une pipeline</u>**

- **Etape 1 :** découpage du texte en segments logiques (par exemple des paragraphes) avec spaCy
- **Etape 2 :** pour chaque chaque segment logique, découpage éventuel en sous-segments si son nombre de tokens est supérieur à $s_{entree}$
- **Etape 3 :** application de Flan-T5 dans chaque segment final, en affinant le prompt se prémunir d'une éventuelle hallucination du modèle
- **Etape 4 :** agrégation des résultas obtenus à l'étape 3

**Note :** spaCy est une blibliothèque Python qui permet de détecter des entités nommées : événements, lieux, individus, etc.

**<u>1.e. Test du modèle de base de Flan-T5</u>**

In [237]:
# !pip install transformers accelerate torch spacy

# à éviter si on peut, car cela génère un message d'erreur, avec derrière un problème pénible de problèmes de dépendances incomaptibles entre elles. 
# Après plusierus tentatives infructueuses, je conclus qu'il vaut mieux éviter de s'embêter avec ça.

In [243]:
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Modèle Flan-T5
MODEL_NAME = "google/flan-t5-base"  # ou flan-t5-large
MAX_TOTAL_TOKENS = 512
RESERVED_TOKENS_FOR_OUTPUT = 100
MAX_INPUT_TOKENS = MAX_TOTAL_TOKENS - RESERVED_TOKENS_FOR_OUTPUT
SEUIL = MAX_INPUT_TOKENS

# Chargement
nlp = spacy.load("fr_core_news_md")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Prompt instructif avec exemples
def build_prompt(segment_text):
    return f"""Tu vas lire un extrait de compte-rendu d'enquête.

Pour chaque événement significatif pour une enquête de police, donne une réponse structurée :
- événement : [résumé en une phrase courte]
- où : [lieu]
- quand : [moment]
- qui : [personnes ou groupes impliqués]

Si aucun événement n'est présent, réponds simplement :
Aucun événement détecté.

Exemples :

Texte :
Le 3 juin 2023, un incendie a été signalé dans un entrepôt à Lyon. Les pompiers sont intervenus vers 23h.
Réponse :
1.
- événement : Incendie dans un entrepôt
- où : Lyon
- quand : 3 juin 2023, vers 23h
- qui : pompiers

Texte :
Le ciel était couvert toute la journée. Aucun fait notable n’a été signalé.
Réponse :
Aucun événement détecté.

Texte :
{segment_text}
Réponse :
"""

# Étape 1 : découpage en paragraphes
def decouper_en_paragraphes(texte):
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

# Étape 2 : redécoupage si trop long
def redécouper_si_necessaire(paragraphe, max_tokens):
    doc = nlp(paragraphe)
    sous_segments = []
    courant = ""
    for sent in doc.sents:
        tentative = (courant + " " + sent.text).strip() if courant else sent.text
        tokens = tokenizer(tentative, return_tensors="pt", truncation=False)["input_ids"].shape[1]
        if tokens <= max_tokens:
            courant = tentative
        else:
            if courant:
                sous_segments.append(courant.strip())
            courant = sent.text
    if courant:
        sous_segments.append(courant.strip())
    return sous_segments

# Étape 3 : traitement Flan-T5
def traiter_segment(segment):
    prompt = build_prompt(segment)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
    outputs = model.generate(
        **inputs,
        max_new_tokens=RESERVED_TOKENS_FOR_OUTPUT,
        num_beams=4,
        early_stopping=True
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip()

# Étape 4 : pipeline globale
def extraire_evenements_depuis_texte(long_texte):
    resultats = []

    paragraphes = decouper_en_paragraphes(long_texte)

    for paragraphe in paragraphes:
        tokens = tokenizer(paragraphe, return_tensors="pt", truncation=False)["input_ids"].shape[1]
        segments = [paragraphe] if tokens <= SEUIL else redécouper_si_necessaire(paragraphe, SEUIL)

        for segment in segments:
            sortie = traiter_segment(segment)
            if sortie.lower().strip() != "aucun événement détecté":
                resultats.append(sortie)

    return resultats


In [245]:
texte

'\nCompte-rendu d’enquête – Rapport préliminaire\n\nObjet : Enquête sur des activités suspectes signalées dans le quartier des Érables (Secteur 5)\n\nDate : 24 juillet 2025\nResponsable de l’enquête : Lieutenant A. Mercier\nDurée de l’enquête : Du 18 au 24 juillet 2025\n\nSuite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte. Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives. Les agents ont constaté la présence récurrente de véhicules immatriculés hors département, ainsi que des échanges brefs entre occupants et des individus arrivant à pied.\n\nUne perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet. Sur place, les enquêteurs ont découvert plusieurs sachets contenant une substance poudreuse suspectée d’être de la cocaïne (analyse en cours), ainsi qu’un t

In [249]:
import time

start = time.time() # on va mesurer le temps d'exécution de cet algorithme

resultats = extraire_evenements_depuis_texte(texte)

for i, r in enumerate(resultats, 1):
    print(f"Événement {i}:\n{r}\n{'-'*40}")

end = time.time()
print(f"Temps d'exécution : {end - start:.2f} secondes")

Événement 1:

----------------------------------------
Événement 2:
Item : Enquête sur les activités suspectes signalées dans le quartier des Érables (Secteur 5)
----------------------------------------
Événement 3:
Date : 24 juillet 2025 Responsable de l’enquête : Lieutenant A. Mercier Durée de l’enquête : 18 au 24 juillet 2025
----------------------------------------
Événement 4:
Une enquête sur le terrain a été ouverte au 17 rue des Marronniers à l'occasion de plusieurs signalements de rivières concernant nocturnes inhabituelles à 17 rue des Marronniers. Les investigations ont commencé par des surveillances discrètes sur une période de cinq nuits consécutives. Les agents ont constaté la présence recurrente de véhicules immatriculées
----------------------------------------
Événement 5:
A perquisition was conducted on 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord de la parquet. Sur le site, enquêteurs ont découvert plusieurs sachets contenant une 

**Bilan : bof bof**

- temps d'exécution très long pour un texte de seulement 20 lignes : presque 1min30s

- les résumés sont des simples copier-coller (partiels) 

- les questions où ? quand ? qui ? restent sans réponse

- certains passages en français, d'autres en anglais : dû au fait que le modèle est surtout entraîné pour l'anglais

- un point positif cependant : le découpage semble des événements semble correct, malgré le fait qu'un événement est vide (événement 1)

**Dans la suite, on reprend la pipeline décrite ci-dessus, mais on teste des variantes de Flan T-5 (autres que le modèle de base).**

**<u>1.f. Le grand modèle Flan-T5 spécialisé sur les dialogues (testé isolément, sans spaCy)</u>**

Pas forcément utile d'avoir un modèle spécialisé sur les dialogues, mais ChatGPT le recommande tout de même, donc on teste.

In [276]:
import torch
from transformers import pipeline
import time

**Sur un dialogue, ça marche pas mal en effet... mais sans GPU c'est vraiment lent (50 s juste pour ça)**

In [282]:
dialogue_text = """Pierre: J’ai oublié ma trousse. Tu peux me prêter un stylo.
Lucie: Tiens.
Pierre: Merci. Tu peux me donner une feuille de papier aussi ?
Lucie: Euh… oui. Tiens.
Pierre: Merci. Ça t’ennuie pas si je regarde avec toi ? J’ai oublié mon livre…
Lucie: Non, pas de problème.
Pierre: Pff. Je ne comprends rien. Tu pourras m’expliquer après le cours ?
Lucie: Oui, si tu veux… On ira au café.
Pierre: Oui… euh non, j’ai oublié mon porte-monnaie.
Lucie: Bon allez ! ce n’est pas grave, je t’invite.
Pierre: Tu es trop gentille.
Lucie: Oui, c’est bien possible."""

In [286]:
start = time.perf_counter()

summarized_text = pipe(dialogue_text, max_length=1024)[0]["summary_text"]  # greedy

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Your max_length is set to 1024, but your input_length is only 219. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=109)
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temps d'exécution : 50.87 secondes


In [292]:
summarized_text

'Pierre a oublié sa trousse, un stylo, une feuille de papier et son livre. Lucie donne à Pierre un stylo, une feuille de papier et une invitation.'

**Essais sur des segment naturels du CRE**

In [652]:
# fonction de génération d'un modèle de résumé 

from transformers import pipeline

def generer_flan_t5(modele):
    pipe = pipeline(
    "summarization",
    model=modele,
    device=device
)
    return(pipe)

In [773]:
# on génère le modèle flan-t5-large-dialogsum-fr

modele = "bofenghuang/flan-t5-large-dialogsum-fr"
pipe = generer_flan_t5(modele)

Device set to use cpu


In [710]:
# la qualité du preprompt d'entraînemnt et le choix des exemples sont essentiels

training_preprompt = f"""Tu vas lire un extrait de compte-rendu d'enquête.

Pour chaque événement relaté dans le texte, donne une réponse structurée sous le format de 4 items :
- événement
- où 
- quand
- qui 

Pour l'item "où", cite toutes les lieux évoqués.
Pour l'item "quand", cite toutes les éléments de dates, jours de semaine et horaires évoqués.
Pour l'item "qui", cite toutes les individus impliqués.

En cas d'information manquante pour un item, ne remplis rien.

Exemples :

Texte :
Le 3 juin 2023, un incendie a été signalé dans un entrepôt à Lyon. Les pompiers sont intervenus vers 23h.
Réponse attendue :
- événement : Incendie dans un entrepôt
- où : Lyon
- quand : 03/06/2025, 23h00
- qui : pompiers

Texte :
Le vendredi 20 juillet 2022, Julien et sa femme Marie partent de leur résidence située au 102 rue des lilas à Paris vers 19h45 pour rejoindre un ami.
Réponse attendue :
- événement : partir de chez soi
- où : Paris, 102 rue des lilas
- quand : vendredi 20/07/2022, 19h45
- qui : Julien, sa femme Marie

Texte :
Il pleuvra sur Nice entre lundi et mercredi, la température sera de 22°C.
Réponse attendue :
- événement : pluie
- où : Nice
- quand : de lundi à mercredi
- qui : 
"""

In [712]:
# le prompt qui finalement sera mis en input de la pipeline

def generer_prompt(texte, preprompt):

    prompt = training_preprompt+f"""

Texte : 
{texte}
Réponse :
"""
    return prompt

In [714]:
segment1 = """
Du 18 au 24 juillet 2025\n\nSuite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte
"""
prompt1 = generer_prompt(texte=segment1, preprompt=training_preprompt)

In [716]:
start = time.perf_counter()

resultat1 = pipe(prompt1, max_length=1024)[0]["summary_text"]  # greedy

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Your max_length is set to 1024, but your input_length is only 492. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=246)
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temps d'exécution : 38.35 secondes


In [718]:
resultat1

'- événement : enquête de terrain - où : 17 rue des Marronniers - quand : 18 au 24 juillet 2025'

**Bilan :**

  - temps d'exécution : 38 s
- qualité du résultat : assez satisfaisante (manque tout de même la réponse au qui)

In [721]:
segment2 = """
Une perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet.
"""
prompt2 = generer_prompt(segment2, training_preprompt)

In [723]:
start = time.perf_counter()

resultat2 = pipe(prompt2, max_length=1024)[0]["summary_text"]  # greedy

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Your max_length is set to 1024, but your input_length is only 484. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=242)
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temps d'exécution : 35.34 secondes


In [725]:
resultat2

'- événement : perquisition - où : parquet - quand : le 23 juillet - qui : officier de police judiciaire'

**<u>Bilan :</u>**

- temps d'exécution : 35 s
-  qualité du résultat : événement, quand, qui : ok ; où : erroné

**Un autre essai**

In [730]:
texte3 = "Ce jeudi en début de soirée, Monsieur Dupont était chez son médecin, quand un homme armé s'est introduit dans le cabinet."
prompt3 = generer_prompt(texte3, training_preprompt)

In [732]:
start = time.perf_counter()

resultat3 = pipe(prompt3, max_length=1024)[0]["summary_text"]  # greedy

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Your max_length is set to 1024, but your input_length is only 473. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=236)
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temps d'exécution : 40.77 secondes


In [734]:
resultat3

"- événement : homme armé s'est introduit dans le cabinet de Monsieur Dupont - où : médecin - quand : jeudi - qui : Monsieur Dupont"

**<u>Bilan :</u>**

- événement : pas mal, même si le modèle se trompe sur le propriétaire du cabinet
- où : ok, mais cabinet du médecin aurait été mieux
- quand : ok
- qui : manquent le médecin (inclus dans le où) et l'homme armé (inclus dans l'événement)

**La réponse de ChatGPT pour ces 3 textes, avec exactement les mêmes prompts**


**Texte 1**

    événement : ouverture d’une enquête de terrain

    où : 17 rue des Marronniers

    quand : du 18 au 24/07/2025

    qui : riverains

**Texte 2**

    événement : perquisition

    où :

    quand : 23/07/2025, 6h00

    qui : officier de police judiciaire, parquet

**Texte 3**

    événement : intrusion d’un homme armé dans un cabinet médical

    où : cabinet médical

    quand : jeudi en début de soirée

    qui : Monsieur Dupont, un homme armé, son médecin

**<u>Bilan global du modèle flan-t5-large-dialogsum-fr :</u>**

- globalement assez qualitatif
- demande certainement un post-traitement de la part de l'enquêteur pour corriger les erreurs ou incomplétudes des items
- le temps d'exécution demeure trop long (souvent 30 à 40 secondes pour une simple phrase) ce qui rend son exploitation difficilement possible pour un service d'enquête

**<center>Le dernier point semble donc disqualifier ce modèle spécialisé par few shot</center>**

**<u>1.g. Retour au modèle flan-t5-base</u>**

Ce modèle présente l'avantage d'être plus rapide. Avec un bon prompt, il est censé être de qualité comparable au modèle plus lourd flan-t5-large-dialogsum selon ChatGPT.

In [758]:
modele = "google/flan-t5-base"
pipe = generer_flan_t5(modele)

Device set to use cpu


In [775]:
start = time.perf_counter()

resultat1 = pipe(prompt1, max_length=1024)[0]["summary_text"]  # greedy

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Your max_length is set to 1024, but your input_length is only 492. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=246)
Both `max_new_tokens` (=256) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temps d'exécution : 59.43 secondes


In [777]:
resultat1

'- événement : enquête de terrain - où : 17 rue des Marronniers - quand : 18 au 24 juillet 2025'

**<u>Bilan :</u>**

  - plus rapide en effet que le modèle flan-t5-large-dialogsum (25 s contre 59 s)
  - mais aussi beaucoup moins qualitatif : la réponse n'est plus du tout structurée, malgré le prompt et tous les exemples

**<center>Ce modèle spécialisé par few shot doit être disqualifié</center>**

**<u>1.h. On essaie le modèle flan-5-large généraliste (sans spaCy)</u>**

In [792]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    tokenizer="google/flan-t5-large",
    device=0 if torch.cuda.is_available() else -1,
)

Device set to use cpu


**On fait un essai avec le premier texte**

In [794]:
start = time.perf_counter()

result = pipe(
    prompt1,
    max_length=150,
    min_length=30,
    do_sample=False,
    num_return_sequences=1
)
print(result[0]["generated_text"])

end = time.perf_counter()

temps = end - start
print(f"Temps d'exécution : {temps:.2f} secondes")

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- événement : 18 au 24 juillet 2025 - où : 17 rue des Marronniers - quand : 18 au 24 juillet 2025
Temps d'exécution : 68.64 secondes


**<u>Bilan :</u>**

- qualité du résultat : peu satisfaisante, l'événement est mal rempli
- temps d'exécution : très lent (68 s)

**<center>Ce modèle spécialisé par few shot doit être disqualifié</center>**

## **Approche 2 : Flan-T5 avec spécialisation par fine-tuning**

**Import des librairies**

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import os

In [ ]:
# -------------------
# CONFIGURATION
# -------------------
MODEL_NAME = "google/flan-t5-base"
DATA_PATH = "evenements_structures_divers_1000.jsonl"  # Ton fichier jsonl
OUTPUT_DIR = "./flan-t5-event-finetuned"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3

In [ ]:
# -------------------
# DATASET CHARGEMENT
# -------------------
dataset = load_dataset("json", data_files={"train": DATA_PATH}, split="train")

# Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_input = tokenizer(
        example["input"], max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length"
    )
    label = tokenizer(
        example["output"], max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length"
    )
    model_input["labels"] = label["input_ids"]
    return model_input

tokenized_dataset = dataset.map(preprocess, batched=True)

In [ ]:
# -------------------
# ENTRAÎNEMENT
# -------------------
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="no",
    save_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_total_limit=1,
    fp16=False,  # True si GPU avec support
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

## **Approche 2 : NLP classique + SpaCy (fr_core_news_md) + règles linguistiques**

**Note :** il s'agit donc d'une approche **hybride**, mêlant un algorithme d'IA avec l'utilisation de règles linguistiques (simples). SpaCy permet de faire du NER (Reconnaissance d'entités nommées), c'est l'outil idéal pour détecter des personnes, des lieux etc. Il est donc très intéressant pour ces travaux.

**Compréhension du français	:** ✔️ SpaCy est entraîné sur corpus francophones

**Exécution locale sur CPU	:** ✔️ Modèles SpaCy optimisés pour CPU (fr_core_news_md, ~40MB)

**Gratuit et open-source :**	✔️ SpaCy + modèles français sont libres

**Extraction de quadruplets structurés	:** ✔️ Via NER + parsing + règles personnalisées

**Fiabilité :**	✔️ Méthode déterministe, testable, explicable

### **2.a. Un premier essai**

In [12]:
import spacy

nlp = spacy.load("fr_core_news_md")

def extract_quadruplets(text):
    doc = nlp(text)
    events = []

    for sent in doc.sents:
        lieu = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moment = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        individus = [ent.text for ent in sent.ents if ent.label_ == "PER"]
        action = [token.lemma_ for token in sent if token.pos_ == "VERB"]

        if action:
            events.append({
                "événement": ", ".join(action),
                "où": ", ".join(lieu),
                "quand": ", ".join(moment),
                "qui": ", ".join(individus)
            })

    return events

In [14]:
texte = "Le 21 juin 2023 à 9h15, Jean Dupont est arrivé au 18 rue des Lilas, Paris. Il a assisté à une réunion confidentielle."
quadruplets = extract_quadruplets(texte)

In [16]:
quadruplets

[{'événement': 'arriver',
  'où': 'rue des Lilas, Paris',
  'quand': '',
  'qui': 'Jean Dupont'},
 {'événement': 'assister', 'où': '', 'quand': '', 'qui': ''}]

**Bilan 2a :** 

**Points positifs :**
- le modèle a bien détecté les deux événements
- il les a correctement identifiés
- il a correctement identitfié le lieu lié au premier événement
- il a correctement identifié la personne liée au premier événement

**Points négatifs :**
- n'identifie pas la date du premier événement
- n'identifie pas la personne associée au deuxième événement (toujours Jean Dupont, donc il ne fait pas le lien entre le "Il" de la seconde phrase et "Jean Dupont" de la première phrase
- les descriptions des événements sont trop succints, ils se réduisent à un verbe, sans complément

**Pour info :** ce bilan est transmis à Copilot pour amélioration de la méthode.

### **2.b. Tentative d'amélioration de la méthode**

**Selon Copilot :**  les points négatifs relevés sont classiques quand on utilise SpaCy “brut” : absence de coreference, résumé minimaliste, perte des expressions temporelles non normées…

**Améliorations apportées dans ce qui suit:**

🌍 Détection des lieux (LOC)

📅 Détection des dates et heures (DATE, TIME + regex)

👤 Identification des individus (PER) avec coreference (via coreferee)

🧠 Reconstitution de la phrase verbale complète (verbe + compléments)

🧪 Ajout d’une validation heuristique (exclut les verbes faibles ou abstraits)

**Note :** la gestion de la coréférence a été d'abord testée en essayant d'importer la librairie **coreferee**. Mais cet import s'est avéré impossible après plusieurs heures de tentatives, en raison d'incompatibilités diverses avec des librairies (ou des versions de librairies) utilisées par ailleurs. De nombreux essais (changer les autres librairies, modifier des versions etc.) ont été entreprises mais aucune n'a fonctionné. Il a donc été décidé de ne pas utiliser la libriairie coreferee).

In [18]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # Modèle SpaCy optimisé pour le français

# 💬 Expressions temporelles enrichies
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """
    return [m.group().strip() for m in re.finditer(date_pattern, sent.text)]

# 🧠 Phrase verbale complète
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            subtree = sorted(token.subtree, key=lambda x: x.i)
            phrase = " ".join([t.text for t in subtree])
            return phrase
    return ""

# 🔎 Filtre des verbes peu informatifs
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Mémorisation de l’individu le plus récent
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person:
            individu.append(last_person)
    return individu

# 🧩 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)

        individu = resolve_pronouns(sent, last_person)

        if individu:
            last_person = individu[-1]  # Mémorise la dernière personne rencontrée

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events


**Un exemple pour tester le code précédent**

In [20]:
texte = """
Compte-rendu d’enquête – Rapport préliminaire

Objet : Enquête sur des activités suspectes signalées dans le quartier des Érables (Secteur 5)

Date : 24 juillet 2025
Responsable de l’enquête : Lieutenant A. Mercier
Durée de l’enquête : Du 18 au 24 juillet 2025

Suite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte. Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives. Les agents ont constaté la présence récurrente de véhicules immatriculés hors département, ainsi que des échanges brefs entre occupants et des individus arrivant à pied.

Une perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet. Sur place, les enquêteurs ont découvert plusieurs sachets contenant une substance poudreuse suspectée d’être de la cocaïne (analyse en cours), ainsi qu’un total de 3 200 € en espèces, deux téléphones portables et un carnet mentionnant des noms et montants.

Trois personnes ont été interpellées sur place et placées en garde à vue. L’une d’entre elles est connue des services pour trafic de stupéfiants. Les auditions sont toujours en cours.

L’enquête se poursuit pour établir les filières d’approvisionnement et identifier d’éventuels complices.
"""

In [22]:
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'signalées dans le quartier des Érables ( Secteur 5 ) \n\n', 'où': 'Érables, Secteur 5', 'quand': ', , , , , , , , , , , , , , , , , , , , le, , , , , , , , , , , , , 24 juillet 2025, , , , , , , , , ', 'qui': 'Date, Responsable'}
{'événement': 'concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers ,', 'où': 'rue des Marronniers', 'quand': ', , 18, , , 24 juillet 2025, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , 17, , , , , , , , , , , , , , , , , , , , ', 'qui': ''}
{'événement': 'Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives .', 'où': '', 'quand': 'Le, , , , , , , , , , , , , , , , , , , , , , , , , , , , , ', 'qui': ''}
{'événement': 'Les agents ont constaté la présence récurrente de véhicules immatriculés hors département , ainsi que des échanges brefs entre occupants et des individus arrivant à pied . \n\n', 'où': '', 'quand': 'Le, , , , , , , , , , , , , , , , , , , 

**Bilan  2b :**

**Points positifs :**
- ok pour la compréhension de à qui il" fait référence dans cet exemple
- ce qui fonctionnait dans la version précédente fonctionne encore

**Points négatifs :**
- plein de virgules inutiles
- apparition de "Mars" non pertinente
- la gestion des références est certainement très peu robuste (exemple : autres pronoms que il ou elle)

### **2c. Gestion des problèmes mentionnés**

**But :**

📅 Dates variées : "le 1er juillet 2023", "juillet 2023", "2023", "01/07/2023", etc.

🔎 Filtrage des doublons et chaînes vides dans les dates

🧹 Nettoyage du champ "quand" pour éviter les valeurs parasites

👤 Remplacement des pronoms par l’entité précédente (type "il" → "Jean Dupont")

In [24]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # modèle français SpaCy

# 🗓️ Expression régulière pour dates variées
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|
           juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """

    MONTHS = {
        "janvier", "février", "mars", "avril", "mai", "juin",
        "juillet", "août", "septembre", "octobre", "novembre", "décembre"
    }

    matches = [m.group().strip() for m in re.finditer(date_pattern, sent.text)]
    # 🧹 filtre les mois isolés (ex : "Mars") et les chaînes vides
    results = [r for r in matches if r and r.lower() not in MONTHS and len(r.strip()) > 2]
    return results

# 🔍 Extraction de la phrase verbale
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            phrase = " ".join([t.text for t in sorted(token.subtree, key=lambda x: x.i)])
            return phrase
    return ""

# ⛔️ Verbes trop génériques
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Logique de co-référence basique
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person and last_person not in individu:
            individu.append(last_person)
    return individu

# 🧠 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]

        # 📅 Récupère dates NLP + regex
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)
        moments = [m for m in moments if m and len(m.strip()) > 2]
        moments = list(set(moments))  # supprime doublons

        individu = resolve_pronouns(sent, last_person)
        if individu:
            last_person = individu[-1]

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events

In [26]:
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'signalées dans le quartier des Érables ( Secteur 5 ) \n\n', 'où': 'Érables, Secteur 5', 'quand': '24 juillet 2025', 'qui': 'Date, Responsable'}
{'événement': 'concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers ,', 'où': 'rue des Marronniers', 'quand': '24 juillet 2025', 'qui': ''}
{'événement': 'Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives .', 'où': '', 'quand': '', 'qui': ''}
{'événement': 'Les agents ont constaté la présence récurrente de véhicules immatriculés hors département , ainsi que des échanges brefs entre occupants et des individus arrivant à pied . \n\n', 'où': '', 'quand': '', 'qui': ''}
{'événement': 'Une perquisition a été menée le 23 juillet à 6h00 , en présence d’ un officier de police judiciaire et avec l’ accord du parquet .', 'où': '', 'quand': 'le 23 juillet', 'qui': ''}
{'événement': 'Sur place , les enquêteurs ont découvert plusieurs sachets contenant une sub

**Bof**

**Bilan 2c :**

**Points positifs :**

- résolution des principaux problèmes évoqués précédemment

**Points négatifs :**
- "le même jour" n'est pas interprété comme une date
- les deux derniers événements ont été fusionnés dans une même phrase
- les résumés des événements ne sont pas très convaincants : encore des copiers-collers

In [13]:
# Exemple d'un système d'IA analysant une requête complexe

nlp = spacy.load('fr_core_news_md')

doc = nlp("Quel est le meilleur endroit pour manger des sushis à Paris ?")

for token in doc:

    print(token.text, token.pos_, token.dep_)

Quel ADJ ROOT
est AUX dep
le DET det
meilleur ADJ amod
endroit NOUN nsubj
pour ADP mark
manger VERB advcl
des DET det
sushis NOUN obj
à ADP case
Paris PROPN obl:mod
? PUNCT punct


### **2d. Des tentatives d'amélioration du résumé**

SpaCy est spécialisé dans la tâche de NER (reconnaissance d'entités nommées : lieux, personnes etc.). L'idée maintenant est de le combiner avec un outil de NLP davantage spécialisé dans le résumé de textes (en français).

**Une tentative avec le modèle de résumé local Text_Summarization**

Ce projet open-source propose 3 méthodes extractives (pas génératives) pour résumer un texte en français :

- Mean Summarization : sélection des phrases les plus représentatives via embeddings

- Clustering Summarization : regroupe les phrases par similarité (K-means)

- Graph Summarization : utilise PageRank sur un graphe de similarité entre phrases

📦 Modèles compatibles :

- CamemBERT
- FlauBERT

**On utilise d'abord des règles purement linguistiques, via des expressions régulières**

In [5]:
import re

def extraire_informations(texte):
    # Définir les motifs pour extraire les informations
    motif_lieu = r"à ([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)"
    motif_annee = r"en (\d{4})"
    motif_personnes = r"\b([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)\b"

    # Trouver les correspondances
    lieux = re.findall(motif_lieu, texte)
    annees = re.findall(motif_annee, texte)
    personnes = re.findall(motif_personnes, texte)

    # Nettoyer les résultats
    lieux = [lieu.strip() for lieu in lieux]
    annees = [annee.strip() for annee in annees]
    personnes = list(set([personne.strip() for personne in personnes if len(personne.strip()) > 2]))

    return lieux, annees, personnes

def generer_resume(texte):
    lieux, annees, personnes = extraire_informations(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(annees))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": annees[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)


{'Événement': 'Événement à Montréal', 'Où': 'Montréal', 'Quand': '1987', 'Qui': 'Claire'}
{'Événement': 'Événement à Lisbonne', 'Où': 'Lisbonne', 'Quand': '1957', 'Qui': 'Alfama'}


**On souhaite maintenant ajouter une surcouche Spacy**

**Note (26/07) :** ce stade l'import de spacy ne fonctionne pas. Le message d'erreur signale un problème de compatibilité binaire entre numpy et une bibliothèque compilée en C, ici h5py, utilisée indirectement par spaCy via thinc. Après quelques essais infructueux pour installer des (versions) des librairies compatibles entre elles, la gestion des dépendances s'avère plus délicate que prévu. On laisse tomber cette approche pour le moment.

In [8]:
!python -m spacy download fr_core_news_sm

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\errors.py", line 3, in <module>
    from .compat import Literal
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\compat.py", line 4, in <module>
    from thinc.util import copy_array
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\__init__.py", line 5, in <module>
    from .config import registry
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\config.py", line 5, in <module>
    from .types import Decorator
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\types.py", line 27, in <module>
    from .compat import cupy, has_cupy
  File "C:\Users\olivi\anaconda

**Voici tout de même le code à exécuter pour générer un résumé en utilisant spaCy (uniquement). Si on parvient à importer spaCy et à faire tourner ce code, on pourra ensuite envisager une approche combinée spaCy + utilisation de règles linguistiques par expressions régulières.**

In [9]:
import spacy

# Charger le modèle français de spaCy
nlp = spacy.load("fr_core_news_sm")

def extraire_entites(texte):
    doc = nlp(texte)

    # Initialiser les listes pour stocker les entités
    lieux = []
    dates = []
    personnes = []

    # Parcourir les entités nommées dans le document
    for ent in doc.ents:
        if ent.label_ == "LOC":  # Lieu
            lieux.append(ent.text)
        elif ent.label_ == "DATE":  # Date
            dates.append(ent.text)
        elif ent.label_ == "PER":  # Personne
            personnes.append(ent.text)

    return lieux, dates, personnes

def generer_resume(texte):
    lieux, dates, personnes = extraire_entites(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(dates))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": dates[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)



OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.